# 🔬 Exploratory Data Analysis (EDA): Anti-Money Laundering (AML) Surveillance
**Graduation Project — Financial Crime Detection & Graph Neural Surveillance**

---
### Executive Summary & What This Notebook Does
Money laundering accounts for an estimated **2% to 5% of global GDP** ($800 billion – $2 trillion USD) annually according to the UNODC.
In banking systems, however, laundering transactions are very rare (**~0.1%** of all transfers, and **3.08%** of accounts).

This notebook explores **206,239 banking accounts** and **55,177 sampled transactions** stored in `aml_system.db` in 5 simple steps:
1. **Class Imbalance, ROC-AUC & PR-AUC:** Why standard accuracy is misleading on skewed data, and how **ROC-AUC** and **PR-AUC** work.
2. **Smurfing / Structuring ($10,000 Rule):** How launderers split big sums into transfers just under the **$10,000 BSA Currency Transaction Report (CTR)** limit.
3. **Payment Channels:** Why **ACH** and **Bitcoin** have the highest laundering rates.
4. **Laundering Patterns (Typologies):** How the 8 laundering ring shapes (Gather-Scatter, Fan-In, Fan-Out, Cycle, Stack, Bipartite, Scatter-Gather, Random) behave.
5. **Graph Correlations:** Why network connections (`ach_tx_count`, `in_degree`, `unique_senders`) beat simple dollar averages at catching launderers.

In [ ]:
# Step 1: Import libraries and connect to the SQLite database (aml_system.db)
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set clean chart styling and default figure size
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Find aml_system.db whether running from notebooks/ or the project root folder
db_path = Path('../aml_system.db').resolve()
if not db_path.exists():
    db_path = Path('aml_system.db').resolve()

# Open database connection
conn = sqlite3.connect(str(db_path))
print(f'[✓] Connected to SQLite database: {db_path}')

---
## 1. Class Imbalance Analysis & The 'Accuracy Fallacy'
First, let's count how many accounts and transactions are legitimate (`0`) vs. laundering (`1`).

In [ ]:
# Count Legitimate (0) vs. Suspicious (1) accounts in the accounts table
acc_summary = pd.read_sql('SELECT is_suspicious, COUNT(*) as count FROM accounts GROUP BY is_suspicious', conn)
acc_summary['pct'] = (acc_summary['count'] / acc_summary['count'].sum() * 100).round(2)
acc_summary['label'] = acc_summary['is_suspicious'].map({0: 'Legitimate Accounts', 1: 'Suspicious Accounts'})

# Count Legitimate (0) vs. Laundering (1) transfers in the transactions table
tx_summary = pd.read_sql('SELECT is_laundering, COUNT(*) as count FROM transactions GROUP BY is_laundering', conn)
tx_summary['pct'] = (tx_summary['count'] / tx_summary['count'].sum() * 100).round(2)
tx_summary['label'] = tx_summary['is_laundering'].map({0: 'Legitimate Transfers', 1: 'Laundering Transfers'})

print('=== ACCOUNT POPULATION ===')
print(acc_summary[['label', 'count', 'pct']])
print('\n=== SAMPLED TRANSACTION POPULATION ===')
print(tx_summary[['label', 'count', 'pct']])

# Plot pie charts comparing account and transaction class ratios
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#10b981', '#ef4444']  # Green = Legitimate, Red = Laundering

ax1.pie(acc_summary['count'], labels=acc_summary['label'], autopct='%1.2f%%', startangle=140, colors=colors, explode=(0, 0.1))
ax1.set_title('Accounts Class Distribution (206,239 Entities)', fontweight='bold')

ax2.pie(tx_summary['count'], labels=tx_summary['label'], autopct='%1.2f%%', startangle=140, colors=colors, explode=(0, 0.1))
ax2.set_title('Sampled Transactions Distribution (55,177 Transfers)', fontweight='bold')

plt.tight_layout()
plt.show()

### Simple Explanation: How ROC-AUC and PR-AUC Work & Why Accuracy Is Removed
In the full dataset, only **~0.1%** of transactions (and **3.08%** of accounts) are laundering.
- **Why Accuracy Fails:** If a bad model simply guesses `Legitimate (0)` for every account, it gets **96.9% accuracy** (or **99.9%** on transactions) while catching **0 launderers**. Single-cutoff metrics (Accuracy, Precision, Recall, F1) also depend on an arbitrary threshold, so we remove them and use **ROC-AUC** and **PR-AUC**.

#### 1. ROC-AUC (Receiver Operating Characteristic Area Under the Curve)
- **What It Calculates:** The chance that a randomly picked laundering account gets a higher risk score than a randomly picked legitimate account: $\text{ROC-AUC} = P(\hat{p}_{\text{laundering}} > \hat{p}_{\text{legitimate}})$.
- **How It Works:** Sweeps the cutoff $t$ from $1.0$ down to $0.0$ and plots **True Positive Rate ($\text{TPR}$)** vs. **False Positive Rate ($\text{FPR}$)**:
  $$\text{TPR}(t) = \frac{TP(t)}{TP(t) + FN(t)}, \qquad \text{FPR}(t) = \frac{FP(t)}{FP(t) + TN(t)}$$

#### 2. PR-AUC (Precision-Recall Area Under the Curve / Average Precision)
- **What It Calculates:** How pure our laundering alerts are (**Precision**) across every capture level (**Recall**), **completely ignoring True Negatives ($TN$)** so the huge number of legitimate accounts cannot inflate the score.
- **How It Works:** Sweeps the cutoff $t$ from $1.0$ down to $0.0$ and integrates **Precision** against **Recall**:
  $$\text{PR-AUC} = \sum_{n=1}^{N} (\text{Recall}_n - \text{Recall}_{n-1}) \times \text{Precision}_n$$
- **How PR-AUC Decides Laundering vs. Legitimate:** Both **XGBoost** (`eval_metric='aucpr'`) and **GraphSAGE** find the best threshold on the Precision-Recall curve and shift it to **50% (`0.50`)**, so any account with calibrated risk $\ge 50\%$ is flagged as laundering.

---
## 2. Financial Distribution & The Structuring ('Smurfing') Signature
Under the US Bank Secrecy Act (BSA, 31 CFR § 1010.311), banks must file a Currency Transaction Report (CTR) for transfers over **$10,000**.
Launderers use **smurfing** (splitting large sums into transfers just under $10,000) to avoid triggering reports.

In [ ]:
# Load transfer amounts for legitimate (0) and laundering (1) transactions
amounts = pd.read_sql('SELECT is_laundering, amount_paid FROM transactions', conn)
legit = amounts[amounts['is_laundering'] == 0]['amount_paid']
laund = amounts[amounts['is_laundering'] == 1]['amount_paid']

# Compare percentiles (5th, 25th, 50th/Median, 75th, 90th, 95th, 99th)
percentiles = [0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
stats_df = pd.DataFrame({
    'Metric / Percentile': ['Count', 'Mean ($)', 'Min ($)', '5th Pct ($)', '25th Pct ($)', 'Median / 50th Pct ($)', '75th Pct ($)', '90th Pct ($)', '95th Pct ($)', '99th Pct ($)', 'Max ($)'],
    'Legitimate Transfers': [
        f'{len(legit):,}', f'{legit.mean():,.2f}', f'{legit.min():,.2f}',
        *[f'{legit.quantile(p):,.2f}' for p in percentiles],
        f'{legit.max():,.2f}'
    ],
    'Laundering Transfers': [
        f'{len(laund):,}', f'{laund.mean():,.2f}', f'{laund.min():,.2f}',
        *[f'{laund.quantile(p):,.2f}' for p in percentiles],
        f'{laund.max():,.2f}'
    ]
})
print(stats_df.to_string(index=False))

# Plot log10 dollar distribution and mark the $10,000 BSA CTR reporting line
fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(np.log10(legit[legit > 0]), color='#10b981', label='Legitimate (Log10 Amount)', kde=True, stat='density', bins=50, ax=ax, alpha=0.4)
sns.histplot(np.log10(laund[laund > 0]), color='#ef4444', label='Laundering (Log10 Amount)', kde=True, stat='density', bins=50, ax=ax, alpha=0.5)

ax.axvline(np.log10(10000), color='black', linestyle='--', linewidth=2, label='$10,000 BSA CTR Threshold (log10=4.0)')
ax.axvline(np.log10(laund.median()), color='#dc2626', linestyle=':', linewidth=2, label=f'Laundering Median (${laund.median():,.2f})')

ax.set_title('Log-Amount Density: Legitimate vs. Laundering Transfers & Smurfing Clustering', fontsize=13, fontweight='bold')
ax.set_xlabel('Log10(Amount Paid in USD)')
ax.set_ylabel('Density')
ax.legend()
plt.show()

---
## 3. Payment Channels & Exploitation Rates
Here we check which payment methods (ACH, Bitcoin, Cash, Cheque, Wire, etc.) are most heavily used by money launderers.

In [ ]:
# Group transactions by payment format and laundering status
pay_query = '''
SELECT payment_format, 
       is_laundering,
       COUNT(*) as tx_count,
       SUM(amount_paid) as total_volume
FROM transactions 
GROUP BY payment_format, is_laundering
'''
pay_df = pd.read_sql(pay_query, conn)

# Calculate the laundering percentage rate for each payment channel
pivot_cnt = pay_df.pivot(index='payment_format', columns='is_laundering', values='tx_count').fillna(0)
pivot_cnt.columns = ['Legitimate', 'Laundering']
pivot_cnt['Total'] = pivot_cnt['Legitimate'] + pivot_cnt['Laundering']
pivot_cnt['Laundering_Rate_%'] = (pivot_cnt['Laundering'] / pivot_cnt['Total'] * 100).round(2)

pivot_cnt = pivot_cnt.sort_values(by='Laundering', ascending=False)
print(pivot_cnt)

# Plot stacked bar chart of Legitimate vs. Laundering transfers per channel
fig, ax = plt.subplots(figsize=(10, 5))
pivot_cnt[['Legitimate', 'Laundering']].plot(kind='bar', stacked=True, color=['#10b981', '#ef4444'], ax=ax)
ax.set_title('Payment Channel Transaction Volume by Legitimacy', fontweight='bold')
ax.set_ylabel('Number of Transactions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 4. Typology Taxonomy & Ring Behavior
We inspect the 8 laundering patterns (Gather-Scatter, Scatter-Gather, Fan-Out, Bipartite, Stack, Fan-In, Cycle, Random) to compare their transaction counts and dollar volumes.

In [ ]:
# Query transaction count and total dollar volume for each laundering pattern
typo_query = '''
SELECT pattern_type, 
       COUNT(*) as tx_count, 
       SUM(amount_paid) as total_volume, 
       AVG(amount_paid) as avg_amount
FROM transactions 
WHERE is_laundering = 1 
GROUP BY pattern_type 
ORDER BY tx_count DESC
'''
typo_df = pd.read_sql(typo_query, conn)
print(typo_df.to_string(index=False))

# Plot transaction counts (left) and total dollar volume (right) for each pattern
sub_typo = typo_df[typo_df['pattern_type'] != 'NONE'].copy()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=sub_typo, x='pattern_type', y='tx_count', palette='mako', ax=ax1)
ax1.set_title('Identified Laundering Transfers by Typology', fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.set_ylabel('Transaction Count')

sns.barplot(data=sub_typo, x='pattern_type', y='total_volume', palette='rocket', ax=ax2)
ax2.set_title('Total Capital Laundered by Typology (USD)', fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.set_ylabel('Total Volume ($)')

plt.tight_layout()
plt.show()

---
## 5. Graph Topology & Correlation Matrix
Finally, we measure how strongly each account feature correlates with being a suspicious account (`is_suspicious`).

In [ ]:
# Load key graph and flow features from the accounts table
corr_query = '''
SELECT in_degree, out_degree, degree_centrality, total_tx_in, total_tx_out,
       avg_amount_in, avg_amount_out, unique_senders, unique_receivers,
       ach_tx_count, wire_tx_count, in_out_tx_ratio, is_suspicious
FROM accounts
'''
corr_df = pd.read_sql(corr_query, conn)

# Compute Pearson correlation matrix (-1.0 to +1.0)
corr_matrix = corr_df.corr()

print('=== CORRELATION WITH IS_SUSPICIOUS ===')
print(corr_matrix['is_suspicious'].sort_values(ascending=False))

# Plot correlation heatmap
plt.figure(figsize=(11, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='vlag', center=0, linewidths=0.5)
plt.title('Correlation Heatmap: Graph Topological vs. Behavioral AML Features', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

---
## 6. Key Takeaways
1. **Graph & Channel Features Beat Raw Dollar Averages:** `ach_tx_count` ($r = +0.36$), `unique_senders` ($r = +0.095$), and `in_degree` ($r = +0.095$) correlate much more strongly with laundering than simple dollar averages ($r \approx 0.01$).
2. **Smurfing Below $10,000:** Laundering transfers have a median of **$8,667.21**, clustering right below the $10,000 BSA reporting threshold.
3. **ACH & Bitcoin Vulnerability:** ACH carries 86.6% of all sampled laundering transfers, while Bitcoin has the highest percentage rate of laundering (73.7%).
4. **Why We Combine XGBoost + GraphSAGE:** Blending **XGBoost** (50% weight) with **GraphSAGE** (50% weight) captures both individual account numbers and multi-hop ring connections, achieving **0.8807 PR-AUC** and **0.9937 ROC-AUC**.